# Udemy Courses Dataset: Preprocessing

## 1. Notebook Goal

This notebook prepares a clean recommendation-ready version of the Udemy Courses dataset. The processed dataset will be used later by the TF-IDF and KNN recommenders, so the goal is to keep the transformation simple, transparent, and reproducible.


## 2. Import Libraries

Only standard data preparation libraries are needed at this stage. Text cleaning is intentionally basic because advanced NLP steps such as stemming or lemmatization are out of scope for the first preprocessing version.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


## 3. Load Raw Dataset

The raw dataset is loaded from `data/raw/udemy_courses.csv`. A fallback path is included so the notebook can still run if it is opened from inside the `notebooks` folder.


In [ ]:
raw_data_path = Path("data/raw/udemy_courses.csv")
if not raw_data_path.exists():
    raw_data_path = Path("../data/raw/udemy_courses.csv")

df_raw = pd.read_csv(raw_data_path)
df_raw.head()


## 4. Select Recommendation Features

The recommender models do not need every column from the raw dataset. We keep the main course identifier, text/category fields, and numeric popularity or course-size indicators. Columns such as URL and publication timestamp are excluded because they are not required for the first recommender versions.


In [ ]:
recommendation_columns = [
    "course_id",
    "course_title",
    "subject",
    "level",
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
]

df = df_raw[recommendation_columns].copy()
df.head()


## 5. Initial Data Quality Snapshot

Before changing the data, we check missing values and duplicates. This gives us a clear before/after comparison for the preprocessing steps.


In [ ]:
initial_quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_share": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
})

print(f"Initial rows: {len(df):,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate course IDs: {df.duplicated(subset='course_id').sum():,}")
initial_quality


## 6. Basic Text Cleaning

The text cleaning is intentionally simple and explainable: lowercase text, remove punctuation that can create noisy tokens, and normalize repeated spaces. This is enough for the first content-based recommender input without introducing advanced NLP complexity.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).lower()
    value = re.sub(r"[^\w+.#\s]", " ", value, flags=re.UNICODE)
    value = re.sub(r"\s+", " ", value).strip()
    return value

text_columns = ["course_title", "subject", "level"]
for column in text_columns:
    df[column] = df[column].apply(clean_text)

df[text_columns].head()


## 7. Handle Missing Values

The current dataset does not contain missing values, but the preprocessing notebook should still define a clear policy. Rows without a course ID or title are not useful for recommendations and are removed. Missing categorical values are labeled as `unknown`, while missing numeric values are replaced with `0` so KNN can later use numeric features without errors.


In [ ]:
df = df.dropna(subset=["course_id", "course_title"]).copy()

categorical_columns = ["subject", "level"]
numeric_columns = ["num_subscribers", "num_reviews", "price", "content_duration"]

for column in categorical_columns:
    df[column] = df[column].fillna("unknown")
    df[column] = df[column].replace("", "unknown")

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)

missing_after_handling = df.isna().sum()
missing_after_handling


## 8. Remove Duplicate Courses

Duplicate rows and duplicate course IDs can bias recommendations because the same course may appear more than once. We first remove exact duplicate rows, then keep the first record for any repeated `course_id`.


In [ ]:
rows_before_deduplication = len(df)

df = df.drop_duplicates().copy()
df = df.drop_duplicates(subset="course_id", keep="first").copy()

rows_after_deduplication = len(df)
removed_rows = rows_before_deduplication - rows_after_deduplication

print(f"Rows before deduplication: {rows_before_deduplication:,}")
print(f"Rows after deduplication: {rows_after_deduplication:,}")
print(f"Rows removed: {removed_rows:,}")


## 9. Create Combined Features

The `combined_features` column joins the main text fields into one model-ready text representation. This column will be the main input for the upcoming TF-IDF recommender and can also support KNN experiments when combined with numeric features.


In [ ]:
df["combined_features"] = (
    df["course_title"].fillna("")
    + " "
    + df["subject"].fillna("")
    + " "
    + df["level"].fillna("")
)

df["combined_features"] = df["combined_features"].apply(clean_text)
df[["course_id", "course_title", "subject", "level", "combined_features"]].head()


## 10. Consistency Checks

These checks confirm that the processed dataset is ready for recommender modeling. We check for empty combined text, remaining duplicates, negative numeric values, and suspicious zero values in important numeric fields.


In [ ]:
consistency_checks = pd.Series({
    "rows": len(df),
    "empty_combined_features": df["combined_features"].eq("").sum(),
    "duplicate_rows": df.duplicated().sum(),
    "duplicate_course_ids": df.duplicated(subset="course_id").sum(),
    "negative_num_subscribers": (df["num_subscribers"] < 0).sum(),
    "negative_num_reviews": (df["num_reviews"] < 0).sum(),
    "negative_price": (df["price"] < 0).sum(),
    "negative_content_duration": (df["content_duration"] < 0).sum(),
    "zero_subscribers": (df["num_subscribers"] == 0).sum(),
    "zero_reviews": (df["num_reviews"] == 0).sum(),
    "zero_price": (df["price"] == 0).sum(),
    "zero_content_duration": (df["content_duration"] == 0).sum(),
})

consistency_checks


The zero-value checks are not automatically treated as errors. Free courses can have a price of `0`, and courses with no subscribers or reviews may still be valid marketplace records. However, these counts are useful to keep visible before modeling.


## 11. Save Processed Dataset

The final processed dataset is saved in `data/processed/processed_udemy_courses.csv`. This gives both recommender notebooks a stable input file and avoids repeating preprocessing logic in multiple places.


In [ ]:
processed_data_path = Path("data/processed/processed_udemy_courses.csv")
if not processed_data_path.parent.exists():
    processed_data_path = Path("../data/processed/processed_udemy_courses.csv")

processed_data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(processed_data_path, index=False)

print(f"Processed dataset saved to: {processed_data_path}")
print(f"Final shape: {df.shape}")


## 12. Final Preview

The final preview confirms that the processed dataset contains the selected recommendation fields plus the new `combined_features` column.


In [ ]:
df.head()
